# Part 0 — Why Vector Databases? The Story

One customer question, one problem, and the chain of ideas that solves it. Each chapter answers the question raised by the previous chapter.

```text
1 PROBLEM       "I forgot my password"  ──►  a keyword search misses the answer
2 WHY           Different words can express the same meaning
3 SIMILARITY    Rank documents by closeness; return the top-K
4 SCALARS       One number
5 VECTORS       A position represented by a list of numbers
6 EMBEDDINGS    A model turns text into a vector
7 CLOSENESS     Compare vectors using a similarity measure
8 VECTOR DB     Store vectors and retrieve the closest records
9 BACK TO THE CUSTOMER   Try the original question again
```

**Learning goal:** By the end, explain in your own words how text becomes an embedding and how a vector database uses it to retrieve relevant records.

**Class plan — about 2.5–3 hours**, including discussion and hands-on work. You do not need a machine-learning or linear-algebra background.

Next parts: 1 Pinecone in depth · 2 the e-commerce knowledge base · 3 measuring search quality.


## Chapter 1 — The problem

A company keeps its help-centre articles in a normal relational database — a table:

```text
help_articles
┌────┬────────────────────────────────────┬──────────┐
│ id │ title                              │ category │
├────┼────────────────────────────────────┼──────────┤
│ 1  │ How to reset my password?          │ account  │
│ 2  │ How to change my account password? │ account  │
│ 3  │ How to update my email address?    │ account  │
│ 4  │ How to track my order?             │ orders   │
│ 5  │ How to cancel my subscription?     │ billing  │
└────┴────────────────────────────────────┴──────────┘
```

A customer types into the help box: **"I forgot my password. How can I get back into my account?"**

The search box runs SQL:

```sql
SELECT * FROM help_articles WHERE title LIKE '%forgot my password%';
-- 0 rows
```

**0 rows.** The answer is sitting right there in row 1 — and the database can't see it.

## Chapter 2 — Why a keyword search can miss it

This query:

```sql
SELECT * FROM help_articles WHERE title LIKE '%forgot my password%';
-- 0 rows
```

asks whether the title contains those characters in that order. Row 1 says **"reset"**, not **"forgot"**, so it does not match.

Real customers describe the same problem in different ways:

| Customer writes | Simple keyword search | What happens |
|---|---|---|
| I forgot my password | `LIKE '%forgot%'` | Misses row 1 — it says "reset" |
| I can't log in anymore | `LIKE '%log in%'` | May miss row 1 — those words are absent |
| I'm locked out of my account | `LIKE '%account%'` | May find an account article, but not necessarily the right one |

The database is not broken. Relational databases are excellent at structured queries:

| Great for | A different kind of question |
|---|---|
| `WHERE id = 1` | Which article means something similar to this question? |
| `WHERE category = 'account'` | Which article is most relevant to what the customer meant? |
| totals, joins, ranges and exact values | Find relevant content even when the wording differs |

Relational databases can also support full-text search, fuzzy matching and, with extensions such as PostgreSQL `pgvector`, vector search. The point here is narrower: **a basic keyword query does not automatically understand meaning.**

> **What we need instead:** not only "which row contains these words?" but also "which rows are most similar in meaning?"


## Chapter 3 — Similarity search

Searching for the items closest in meaning has a name: **similarity search**.

```text
question ──► compare with articles ──► rank closest first ──► return top-K
```

Imagine a search gives these illustrative scores:

| Rank | Help article | Example score |
|---:|---|---:|
| 1 | How to reset my password? | 0.92 |
| 2 | How to change my account password? | 0.88 |
| 3 | How to update my email address? | 0.71 |
| 4 | How to cancel my subscription? | 0.35 |
| 5 | How to track my order? | 0.21 |

These numbers are **made-up examples** to explain ranking. Real scores depend on the model, data and metric.

**Top-K** means how many results to return:
- `top_k=1` → return the best one.
- `top_k=3` → return the best three.

But how can a computer compare meaning? We first represent each piece of text using numbers. That brings us to vectors.

> **Next question:** how do we represent something using numbers?


## Chapter 4 — Scalars: one number

A **scalar** is simply one numerical value.

```python
age = 30
temperature = 35
price = 500

print(age)
```

Each variable above contains one number. That is all we need to know about scalars for this lesson.

> **Next question:** what if one number is not enough to describe something?


In [ ]:
age = 30
temperature = 35
price = 500

print(age)

## Chapter 5 — Vectors: a position written as numbers

Imagine a map. A location can be described using two coordinates:

```text
 y
40 ┤
30 ┤                    • market [30, 25]
20 ┤      • home [10, 20]
   │        • school [12, 18]
10 ┤
 0 ┼──────┬──────┬──────┬──────┬── x
   0     10     20     30     40
```

`home = [10, 20]` means 10 units across and 20 units up. In this simple example, the two numbers locate a point.

```python
# A location represented using two numbers
home = [10, 20]

print("X coordinate:", home[0])
print("Y coordinate:", home[1])
print("Dimensions:", len(home))
```

Home and school have nearby coordinates, so they are close on this map.

| Word | Beginner-friendly meaning | Example |
|---|---|---|
| **Scalar** | One number | `30` |
| **Vector** | A list of numbers that represents something | `[10, 20]` |
| **Dimension** | How many numbers are in the vector | `2` |

A vector can have 2, 3, or many more dimensions. For example, `[90, 85, 80]` has 3 dimensions.

> **Next question:** can we represent a **sentence** using a list of numbers too?


## Chapter 6 — Embeddings: text turned into numbers

An **embedding** is a vector generated from data such as text. An **embedding model** creates it.

```text
"How to reset my password?" ──► embedding model ──► [0.0123, -0.0441, ...]
"I forgot my password..."   ──► embedding model ──► [0.0119, -0.0398, ...]
"How to track my order?"    ──► embedding model ──► [-0.0302, 0.0217, ...]
```

The example numbers above are **illustrative**, not actual output. The model is trained beforehand to learn patterns in language. When we send it text, it produces a numerical representation based on what it learned.

- **We do not manually choose the numbers.** The embedding model generates them.
- **We do not normally know what one individual dimension means.** The useful information is represented across the vector as a whole.
- Similar meaning often leads to similar embeddings, but this is a learned pattern—not a guarantee that every relevant result will rank first.
- For `text-embedding-3-small`, the default output has **1536 values**. The dimension is determined by the model and its configuration; other models can return different dimensions.

### Tokens and the input limit

A **token** is a piece of text that a model processes. It may be a whole word, part of a word, or punctuation.
Every embedding model has a maximum number of tokens it reads per text — and models behave differently when the
text is longer:

| Model | Limit per text | Text longer than the limit |
|---|---|---|
| OpenAI `text-embedding-3-small` (ours) | 8,191 tokens | The request **fails with an error** — nothing is cut for you |
| `all-MiniLM-L6-v2` (a popular free model) | 256 word pieces | The rest is **cut off silently** — no error, and that part is never searchable |

Cutting text to the limit yourself (**truncation**) avoids the error, but whatever comes after the cut is lost.
So long documents are split into smaller **chunks**: each chunk is embedded and stored as its own record. Chunking
is an important topic for the next stage of RAG learning.

### Things to know before using an embedding model

| Concept | What it means |
|---|---|
| Same model | Use the same embedding model for stored articles and incoming questions. If you change models, re-embed the stored content. |
| Matching dimensions | The vector dimension must match the dimension configured in the vector index. |
| Company jargon | A general model may not know what an internal term means. Add useful context or definitions to the text when needed. |
| Languages | Cross-language search quality varies by model. Test Telugu, English and Hinglish queries rather than assuming they will work equally well. |
| Cost | Embedding APIs are usage-priced per token. Check the provider's current pricing before estimating project costs. |
| Size | Each stored vector takes space: 1536 numbers × 4 bytes ≈ **6 KB**, so a million vectors ≈ 6 GB. A model with more dimensions needs more space. |

> **Next question:** once text is represented as numbers, how do we compare two embeddings?


### Setup

Keys go in a `.env` file next to this notebook — never paste secrets into a notebook cell:

```text
OPENAI_API_KEY=your-openai-key
PINECONE_API_KEY=your-pinecone-key
```

We need the OpenAI key for the embedding exercise. We will check the Pinecone key later, when we reach the database section.


In [ ]:
%pip install -q openai pinecone python-dotenv

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup: keys ──► OpenAI client ──► embed() helper
#
# Does NOT connect to Pinecone yet (chapter 8).
# ─────────────────────────────────────────────────────────────
import os
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
for key in ("OPENAI_API_KEY",):
    if not os.environ.get(key):
        raise ValueError(f"Please set {key} in your .env file.")

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"          # one model, used for articles AND questions


def embed(texts):
    """List of texts in → list of embeddings (1536 numbers each) out, same order."""
    response = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]

In [ ]:
# ─────────────────────────────────────────────────────────────
# Turn the 5 help articles into embeddings
#
#   5 titles ──► one call ──► 5 embeddings ──► print dimension + first 5 numbers
#
# Does NOT store anything yet. Chapter 8 does.
# ─────────────────────────────────────────────────────────────
articles = [
    "How to reset my password?",
    "How to change my account password?",
    "How to update my email address?",
    "How to track my order?",
    "How to cancel my subscription?",
]
categories = ["account", "account", "account", "orders", "billing"]

article_vectors = embed(articles)
for text, vector in zip(articles, article_vectors):
    print(f"{text:38} {len(vector)} numbers   [{', '.join(f'{x:+.4f}' for x in vector[:5])}, …]")

Every article becomes exactly **1536 numbers** — the default output dimension of `text-embedding-3-small`.

You cannot usually read the meaning from one number. The useful signal is in the vector as a whole and in how it
compares with another vector.

> **Next question:** how do we measure how close two embeddings are?

## Chapter 7 — Measuring closeness: cosine similarity

Imagine each vector as an arrow starting at zero. **Cosine similarity compares the direction of two arrows.**

- If the arrows point in nearly the same direction, the cosine similarity is high.
- If they are at a right angle, the cosine similarity is 0.
- If they point in opposite directions, the cosine similarity is -1.

| Cosine score | Geometric interpretation |
|---:|---|
| `1` | Same direction |
| `0` | Perpendicular directions |
| `-1` | Opposite directions |

For text embeddings, a higher score often indicates a closer match **within the model and search setup**. It is not a universal measure of meaning.

The formula is:

```text
cosine similarity = (A · B) / (length of A × length of B)
```

You do not need to calculate this by hand in this lesson. The embedding/search libraries do the calculation.

**Remember:**
- Real scores can bunch together. A good match might be 0.6 rather than 0.99.
- A score is **not a probability**. `0.80` does not mean "80% sure".
- The highest-scoring result is the closest result found—not a guarantee that it is correct.

> **Next question:** how do we store many embeddings and search them efficiently?


In [ ]:
import math

vector1 = [0.9, 0.8, 0.1, 0.7, 0.6]
vector2 = [0.8, 0.9, 0.2, 0.8, 0.5]

# Step 1: Calculate dot product
dot_product = sum(a * b for a, b in zip(vector1, vector2))

# Step 2: Calculate magnitude of each vector
magnitude1 = math.sqrt(sum(a * a for a in vector1))
magnitude2 = math.sqrt(sum(b * b for b in vector2))

# Step 3: Calculate cosine similarity
similarity = dot_product / (magnitude1 * magnitude2)

print("Dot Product:", round(dot_product, 3))
print("Magnitude 1:", round(magnitude1, 3))
print("Magnitude 2:", round(magnitude2, 3))
print("Cosine Similarity:", round(similarity, 3))

In [1]:
import math

vector1 = [0.9, 0.1, 0.1, 0.2, 0.1]
vector2 = [0.1, 0.9, 0.8, 0.1, 0.2]

# Step 1: Calculate dot product
dot_product = sum(a * b for a, b in zip(vector1, vector2))

# Step 2: Calculate magnitude of each vector
magnitude1 = math.sqrt(sum(a * a for a in vector1))
magnitude2 = math.sqrt(sum(b * b for b in vector2))

# Step 3: Calculate cosine similarity
similarity = dot_product / (magnitude1 * magnitude2)

print("Dot Product:", round(dot_product, 3))
print("Magnitude 1:", round(magnitude1, 3))
print("Magnitude 2:", round(magnitude2, 3))
print("Cosine Similarity:", round(similarity, 3))

Dot Product: 0.3
Magnitude 1: 0.938
Magnitude 2: 1.229
Cosine Similarity: 0.26


## Chapter 8 — The vector database

Why not store the embeddings in the relational table from chapter 1? A table can hold a list of numbers, but its
normal indexes are built for exact lookups such as `WHERE id = 1`. Finding the **closest** of millions of vectors
needs an index built for that job — in a vector database, or in an extension such as PostgreSQL `pgvector`.

First, separate two jobs:

| Embedding model | Vector database |
|---|---|
| Converts text into vectors | Stores vectors and associated records |
| Generates numerical representations | Searches for nearby vectors |
| Example: OpenAI `text-embedding-3-small` | Example: Pinecone |

```text
Articles / question
       │
       ▼
Embedding model  ── creates vectors ──►  Vector database
                                          stores and searches
                                          vectors + metadata
```

A **vector database** is designed to store vector representations and retrieve records whose vectors are most similar to a query vector.

### What does one record look like?

```python
record = {
    "id": "article_1",                 # unique identifier
    "values": [0.12, 0.45, 0.78],      # short example vector
    "metadata": {
        "category": "account",         # useful for filtering
        "text": "How to reset my password?"
    }
}
```

The three values above are only a tiny illustration. Our actual model returns 1536 values.

| Field | Meaning |
|---|---|
| `id` | Unique identifier for the record |
| `values` | The embedding vector used for similarity search |
| `metadata` | Extra information, such as category, source or text |

In this notebook, we put the original text in metadata so we can display it with the search result. Other systems may store the original text elsewhere and keep a reference in metadata.

### Three Pinecone terms

| Pinecone word | Beginner-friendly meaning |
|---|---|
| **Index** | The vector collection. Its dimension and similarity metric are configured when it is created. |
| **Namespace** | A logical section inside an index, useful for separating datasets. |
| **Upsert** | Insert a record, or replace the existing record if the same ID is already present. |

For example, one index could contain separate namespaces for `help-articles`, `hr-documents` and `product-catalog`.

### What is metadata filtering?

First find similar vectors, but only among records that meet a condition—for example, only records where `category = "account"`. This narrows the search to the records you want.


We will store these five help articles in a namespace called `help-articles`. A namespace is a logical partition inside the index. A search specifies which namespace to search, so records in other namespaces are kept separate from this search.

The index dimension must match the embedding dimension (1536), and the metric is set to cosine.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Store the 5 articles in Pinecone
#
# Step 1: Check whether the Pinecone index already exists.
# Step 2: If it doesn't exist, create a new index.
# Step 3: Prepare the 5 records with ID, embedding and metadata.
# Step 4: Upload the records using upsert().
# Step 5: Wait until the records become searchable.
#
# IMPORTANT:
# We are reusing the embeddings generated in Chapter 6.
# We do NOT call OpenAI again to generate embeddings.
# ─────────────────────────────────────────────────────────────

from pinecone import Pinecone, ServerlessSpec

# Step 1: Check whether the Pinecone API key is available.
# The API key is required to connect to Pinecone.
if not os.environ.get("PINECONE_API_KEY"):
    raise ValueError(
        "Please add PINECONE_API_KEY to your .env file before this section."
    )

# Step 2: Connect to Pinecone using the API key.
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Define the index name and namespace.
# Index: A collection that stores vectors.
# Namespace: A logical grouping of vectors inside an index.
INDEX_NAME, NAMESPACE = "ecom-kb-tutorial", "help-articles"

# Step 3: Check whether the index already exists.
if not pc.has_index(INDEX_NAME):

    # If the index doesn't exist, create it.
    pc.create_index(
        name=INDEX_NAME,

        # Each embedding contains 1536 numerical values.
        # Therefore, the vector dimension must be 1536.
        dimension=1536,

        # Cosine similarity will be used to compare vectors.
        metric="cosine",

        # Configure Pinecone to use AWS in the us-east-1 region.
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Step 4: Get a reference to the index.
# We will use this object to upload and search vectors.
index = pc.Index(INDEX_NAME)

# Step 5: Prepare the records that need to be stored.
records = [
    {
        # Give each article a unique ID.
        "id": f"article_{number}",

        # Store the embedding generated in Chapter 6.
        "values": vector,

        # Store additional information along with the embedding.
        # Metadata helps us retrieve the original article later.
        "metadata": {
            "category": category,
            "text": text
        }
    }

    # Combine article text, its embedding and category.
    for number, (text, vector, category)
    in enumerate(zip(articles, article_vectors, categories), start=1)
]

# Step 6: Upload all 5 records into Pinecone.
# upsert() inserts new records or updates existing records
# when the same ID is already present.
# namespace groups these records under "help-articles".
index.upsert(
    vectors=records,
    namespace=NAMESPACE
)

# Step 7: Wait until Pinecone finishes indexing the records.
# Newly uploaded records may take a few seconds to become searchable.

for _ in range(30):

    # Get the statistics for our namespace.
    # This tells us how many vectors have been stored.
    found = index.describe_index_stats().namespaces.get(NAMESPACE)

    # Check whether all 5 records are available.
    if found and found.vector_count >= len(records):
        break

    # Wait for 2 seconds before checking again.
    time.sleep(2)

# Step 8: Display the total number of records stored.
print(
    f"{NAMESPACE}: "
    f"{found.vector_count if found else 0} records stored"
)

## Chapter 9 — Back to the customer

The same question the keyword search couldn't answer in chapters 1–2. Embed it with the **same model**, ask Pinecone for the top-K.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Search articles in Pinecone using semantic similarity.
#
# Step 1: Accept a question from the user.
# Step 2: Convert the question into an embedding using the same model.
# Step 3: Send the embedding to Pinecone.
# Step 4: Find the top-K most similar vectors.
# Step 5: Retrieve and display the matching article details.
#
# IMPORTANT:
# We use the same embedding model that we used in Chapter 6
# to generate the article embeddings.
#
# Search is restricted to the "help-articles" namespace.
# ─────────────────────────────────────────────────────────────

def search(question, top_k=3, category=None):

    # Step 1: Convert the user's question into an embedding.
    # embed() converts text into a numerical vector.
    # [0] extracts the first (and only) embedding from the result.
    question_vector = embed([question])[0]

    # Step 2: Search Pinecone for the closest matching vectors.
    result = index.query(

        # Numerical representation of the user's question.
        vector=question_vector,

        # Return the top-K closest matching records.
        # For example, top_k=3 returns the three closest matches.
        top_k=top_k,

        # Search only within this namespace.
        namespace=NAMESPACE,

        # Include metadata such as article text and category
        # in the search results.
        include_metadata=True,

        # Optional metadata filter.
        # If a category is provided, return only records
        # belonging to that category.
        # If category is None, search across all categories.
        filter={"category": {"$eq": category}} if category else None,
    )

    # Step 3: Display the user's original question.
    print(f"QUESTION: {question}  |  top_k={top_k}")

    # Step 4: Iterate through the matching records.
    for match in result.matches:

        # Extract metadata from the matching record.
        # Use an empty dictionary if metadata is unavailable.
        metadata = match.metadata or {}

        # Retrieve the article category.
        category_name = metadata.get('category', 'unknown')

        # Retrieve the original article text.
        text_value = metadata.get(
            'text',
            '(text not stored in metadata)'
        )

        # Display:
        # match.score = Cosine similarity score.
        # match.id    = Unique ID of the matching article.
        # category    = Article category.
        # text        = Original article content.
        print(
            f"   {match.score:.3f}  "
            f"{match.id}  "
            f"[{category_name}]  "
            f"{text_value}"
        )

    print()


# ─────────────────────────────────────────────────────────────
# Example 1: User asks about resetting a forgotten password.
# ─────────────────────────────────────────────────────────────
search("I forgot my password. How can I get back into my account?")


# ─────────────────────────────────────────────────────────────
# Example 2: User asks about login problems using different words.
# ─────────────────────────────────────────────────────────────
search("I can't log in anymore")


# ─────────────────────────────────────────────────────────────
# Example 3: User describes being locked out of their account.
# ─────────────────────────────────────────────────────────────
search("I'm locked out of my account")

Compare with chapter 2: the questions the keyword search missed or got wrong should now return the password articles at the top —
**without sharing the words "forgot", "log in" or "locked"**. That is similarity search over embeddings.

Two more things the vector database does:

In [ ]:
search("I forgot my password", top_k=1)             # Return only the best match
search("I forgot my password", category="billing")   # Search only billing records


The filter returns the best **billing** article even though it may be a poor match for a password question. A vector database returns the closest records available under the search settings; it does **not** know whether the result truly answers the question.

In a real application, we may add a score threshold, reranking, metadata filters or other checks. These techniques can improve retrieval, but none guarantees a correct answer.

## Try your own — predict before you run

For each query:
1. Write down which article you expect to appear first.
2. Explain why.
3. Run the search and compare the result with your prediction.

Try changing `top_k` and the category filter too.


In [ ]:
search("I want to stop paying every month")
search("Where is my parcel?")
search("నేను నా పాస్‌వర్డ్ మర్చిపోయాను")             # Telugu: "I forgot my password"
search("mera password bhool gaya")              # Hinglish

## The story in one breath

**A keyword search** can miss a relevant article when the wording differs → we want **similarity search** → similarity search compares numerical representations → a **vector** is a list of numbers → an **embedding model** turns text into a vector → a similarity measure such as **cosine similarity** compares vectors → a **vector database** stores vectors with metadata and retrieves the closest records → we inspect the results because closest does not always mean correct.


## Quick knowledge check

Try answering these without looking back:

1. What is a scalar?
2. How many dimensions does `[10, 20, 30]` have?
3. Who generates an embedding?
4. Does one embedding dimension have an easy human-readable meaning?
5. What does `top_k=3` ask the database to return?
6. What is the difference between an embedding model and a vector database?
7. What are the three main parts of the Pinecone record in this example?
8. Does the highest similarity score guarantee that the document is correct?
9. Why might we split a long document into chunks?
10. Why should we use the same embedding model for stored documents and incoming questions?

The answers are in the next cell — try first.

## Answers

| # | Answer |
|---|---|
| 1 | One number, such as `30` |
| 2 | 3 |
| 3 | The embedding model — we do not choose the numbers |
| 4 | No. The meaning is spread across the whole vector |
| 5 | Up to the three closest records |
| 6 | The embedding model turns text into vectors; the vector database stores vectors with metadata and finds the closest ones |
| 7 | `id`, `values` (the vector) and `metadata` |
| 8 | No. It is the closest result found, not a guarantee that it is correct |
| 9 | Models have a token limit per text (8,191 for ours — longer text fails or is silently cut, depending on the model), and smaller chunks each stay about one thing |
| 10 | Each model produces its own numbers; vectors from different models can't be compared. Changing the model means re-embedding the stored content |

**Next — Part 1:** the same index, in depth.